# Corporate Reputation: Run all five extractors

**One Google Drive mount • One ticker • One inclusive date range • Separate source outputs.**

1. Copy the *contents* of the framework ZIP into `MyDrive/ColabNotebooks/DataForReputation/`.
2. For YouTube, add Colab Secret `YOUTUBE_API_KEY` with notebook access enabled. For SEC, add `SEC_USER_AGENT` (descriptive identity + real email).
3. Run cells top-to-bottom. Review `run_summary.csv` and `manifest.json` for partial or failed sources.
4. The framework NEVER sends raw datasets to GitHub; see the other notebook for code-only publishing.

**Research caution:** YouTube search finds videos published within the period, then filters their comments by publication date. SEC filings indicate disclosures, not necessarily a reputational event. Yahoo stock bars are daily, while Google Trends frequency varies.

In [1]:
# 1. Mount Drive ONCE; point Python at the framework folder.
from google.colab import drive
from pathlib import Path
import sys

drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/ColabNotebooks/DataForReputation')
if not (PROJECT_ROOT / 'reputation_etl').is_dir():
    raise FileNotFoundError(f'Copy the framework files to {PROJECT_ROOT} first')
sys.path.insert(0, str(PROJECT_ROOT))
print('Framework folder:', PROJECT_ROOT)

Mounted at /content/drive
Framework folder: /content/drive/MyDrive/ColabNotebooks/DataForReputation


3. Parameters
Edit these values once. If you lack access to an API, remove that source from SOURCES until configured. No keys are typed into code cells. If the company-name lookup is unreliable, provide a verified name override.

In [16]:
from google.colab import userdata
from reputation_etl import ExtractionConfig, run_pipeline

def get_secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

#TICKER = 'MSFT'
#START_DATE = '2026-09-01'  # inclusive
#END_DATE = '2026-10-01'    # inclusive
#COMPANY_NAME_OVERRIDE = 'Microsoft Inc.'  # None to resolve via yfinance

TICKER = input("Enter the Yahoo Finance ticker, for example MSFT:  ").strip().upper()
START_DATE = input("Enter start date in YYYY-MM-DD format,for example 2026-09-01:  ").strip()
END_DATE = input("Enter end date in YYYY-MM-DD format,for example 2026-10-01:  ").strip()
COMPANY_NAME_OVERRIDE = input("Enter company name,for example Microsoft Inc.:  ").strip()

# Remove a source if its credentials/approval are not available.
SOURCES = ('gdelt', 'google_trends', 'youtube', 'sec_edgar', 'yahoo_finance')
#SOURCES = ('sec_edgar', 'yahoo_finance')

config = ExtractionConfig(
    ticker=TICKER, start_date=START_DATE, end_date=END_DATE,
    sources=SOURCES, output_root=PROJECT_ROOT,
    company_name_override=COMPANY_NAME_OVERRIDE,
    gdelt_aliases=('Microsoft',) if TICKER == 'MSFT' else (),
    gdelt_window_days=1,
    trends_geo='US',
    trends_search_term=None,
    youtube_api_key=get_secret('YOUTUBE_API_KEY'),
    youtube_max_videos=100, #The YouTube Search API consumes 100 units of quota per search page (up to 50 videos).
    #If you need broader coverage, you can increase this to 100 or 200.
    youtube_max_comment_pages_per_video=10,
    youtube_include_comment_text=False, # must explicitly opt-in if required (switch this to True to acquire the actual message text.)
    youtube_include_author_fields=False,
    sec_user_agent=get_secret('SEC_USER_AGENT'),
)
print('Ticker:', config.ticker, '| dates:', config.start_date, 'to', config.end_date)
print('Selected:', ', '.join(config.sources))
print('YouTube key configured:', bool(config.youtube_api_key))
print('SEC user agent configured:', bool(config.sec_user_agent))

Enter the Yahoo Finance ticker, for example MSFT:  MSFT
Enter start date in YYYY-MM-DD format,for example 2026-09-01:  2026-09-01
Enter end date in YYYY-MM-DD format,for example 2026-10-01:  2026-10-01
Enter company name,for example Microsoft Inc.:  Microsoft Inc.
Ticker: MSFT | dates: 2026-09-01 to 2026-10-01
Selected: sec_edgar, yahoo_finance
YouTube key configured: True
SEC user agent configured: True


In [17]:
# 4. Run extraction. Sources fail independently; inspect statuses afterward.
summary, run_directory = run_pipeline(config)
display(summary)
print('Saved to:', run_directory)

Company: MSFT — Microsoft Inc.

Running sec_edgar ...
sec_edgar: success — 1 rows

Running yahoo_finance ...
yahoo_finance: success — 22 rows

Outputs: /content/drive/MyDrive/ColabNotebooks/DataForReputation/runs/MSFT/2026-09-01_to_2026-10-01/20261104T140311Z_8851f681


,source,status,rows,error,notes
0,sec_edgar,success,1,,SEC filings are independent corroborating disc...
1,yahoo_finance,success,22,,yfinance end parameter converted to inclusive ...


Saved to: /content/drive/MyDrive/ColabNotebooks/DataForReputation/runs/MSFT/2026-09-01_to_2026-10-01/20261104T140311Z_8851f681


In [18]:
# 5. Inspect saved files and manifest (no API keys stored in manifest).
import json
manifest = json.loads((run_directory / 'manifest.json').read_text())
for source, info in manifest['sources'].items():
    print(f"{source:16} {info['status']:8} rows={info['rows']:>6}")
    for note in info.get('notes', []):
        print('   NOTE:', note)
print('\nOutput files:')
for file in sorted(run_directory.iterdir()):
    print(' -', file.name)

sec_edgar        success  rows=     1
   NOTE: SEC filings are independent corroborating disclosure records, NOT validated incident labels.
   NOTE: FilingDate is date-level; acceptanceDateTime can be more precise when available.
   NOTE: SEC submission history is composed of recent AND older index files.
yahoo_finance    success  rows=    22
   NOTE: yfinance end parameter converted to inclusive end date.
   NOTE: Daily stock bars are not hourly prediction signals; market data are external corroboration.
   NOTE: Rolling 7/30-trading-day variables are NaN near the start unless prior warm-up history is fetched.
   NOTE: yfinance is unofficial; verify data license and replace with licensed research data if needed.

Output files:
 - manifest.json
 - run_summary.csv
 - sec_edgar.csv
 - sec_edgar.parquet
 - sec_edgar_historical_file_log.csv
 - sec_edgar_historical_file_log.parquet
 - yahoo_finance.csv
 - yahoo_finance.parquet
 - yahoo_finance_company_metadata.csv
 - yahoo_finance_company_m

## Before interpreting the data

A **partial** status means a source may be incomplete. An **empty** dataset is not proof of no reputational activity. Always inspect API constraints, source timestamps, selection biases, and the request logs. Preserve raw native-time observations before aggregating to hourly features; stock prices and historical search interest do not necessarily have hourly cadence.